# Wessex Stage3G: bounded CPU development reproduction

Version: `stage3g_transfer_reliability_v1`. Treated effluent **daily volume**, initial 30/90/180 calendar-day histories, and the **third day itself** for h=3. This notebook continues only the frozen new development matrix; it does not access final 2024–2025 or protected-site performance, add models/source pools/weather, or recalculate old stages.

The experiment and small CPU checks were run locally. Live Colab login, Google Drive mounting and Drive persistence have **not** been validated by this delivery. Use Drive only if you have chosen to authorize it in your own browser session. All saved predictions and failures remain on the chosen project path. Software completion and scientific benefit are separate statuses.


In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, subprocess, sys, time, zipfile, tempfile, shutil
VERSION = 'stage3g_transfer_reliability_v1'
USE_GOOGLE_DRIVE = False  # optional: change to True in your own Colab session
RUN_FULL_FROZEN = False  # optional continuation of the same approved matrix only
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TASK_STORAGE = Path('/content/drive/MyDrive/WessexStage3G')
else:
    TASK_STORAGE = Path('/content/WessexStage3G')
PACKAGE_DIR = TASK_STORAGE/'packages'
PROJECT_ROOT = TASK_STORAGE/'project'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print('Copy Stage3G ZIP files and the outer SHA256SUMS to:', PACKAGE_DIR)
print('Persistent project directory:', PROJECT_ROOT)
print('Google Drive mounting/persistence is not claimed as tested in the original delivery.')


## Verify every imported package

This importer checks the outer SHA, ZIP CRC, member SHA, declared gzip originals and safe paths. Different existing scientific files are never overwritten. Package receipts are kept separately so independently extractable ZIP parts do not overwrite one another's manifests. Select the code package for a fresh small run; add prediction/diagnostic parts for a cache review. Do not confuse a verified cached replay with fresh training.


In [ ]:
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda:f.read(1048576),b''):h.update(part)
    return h.hexdigest()

def safe_member(name):
    from pathlib import PurePosixPath
    p=PurePosixPath(name)
    if p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name:
        raise ValueError('Unsafe archive path: '+name)
    return p

outer_path=PACKAGE_DIR/'SHA256SUMS'
if not outer_path.is_file():raise FileNotFoundError('Copy the delivery outer SHA256SUMS before importing.')
outer={}
for line in outer_path.read_text().splitlines():
    if line.strip():
        h,name=line.split('  ',1);outer[name]=h
archives=sorted(PACKAGE_DIR.glob('Wessex_Stage3G_*.zip'))
if not archives:raise FileNotFoundError('No Stage3G ZIP packages found.')
verified={};receipts=[]
with tempfile.TemporaryDirectory(prefix='stage3g_import_',dir=PROJECT_ROOT.parent) as temporary:
    staging=Path(temporary)
    for archive in archives:
        if archive.stat().st_size >= 25*1024**2:raise ValueError('Package reaches the 25 MiB limit.')
        if outer.get(archive.name)!=digest(archive):raise ValueError('Outer SHA mismatch: '+archive.name)
        with zipfile.ZipFile(archive) as z:
            names=z.namelist()
            if len(names)!=len(set(names)) or z.testzip() is not None:raise ValueError('ZIP duplicate/CRC failure')
            for info in z.infolist():
                safe_member(info.filename)
                if (info.external_attr>>16)&0o170000==0o120000:raise ValueError('Symlink member rejected')
            candidates=[]
            for name in names:
                if name.endswith('MANIFEST.json'):
                    candidate=json.loads(z.read(name))
                    payload=candidate.get('members_sha256',candidate.get('files_sha256'))
                    if isinstance(payload,dict):candidates.append((name,candidate,payload))
            if len(candidates)!=1:raise ValueError('One member-SHA package manifest is required')
            manifest_name,manifest,payload=candidates[0]
            auxiliaries={manifest_name}|{n for n in names if n.endswith('SHA256SUMS')}
            if set(names)!=set(payload)|auxiliaries:raise ValueError('Unmanifested/missing package member')
            for name,h in payload.items():
                blob=z.read(name)
                if hashlib.sha256(blob).hexdigest()!=h:raise ValueError('Member SHA mismatch: '+name)
                if name in verified and verified[name]!=h:raise ValueError('Conflicting packages: '+name)
                dst=staging/safe_member(name);dst.parent.mkdir(parents=True,exist_ok=True);dst.write_bytes(blob)
                verified[name]=h
            for packed,record in manifest.get('gzip_originals',{}).items():
                import gzip
                original=record['original_path'];safe_member(original)
                if record['original_bytes']>1024**3:raise ValueError('Excessive declared original')
                blob=gzip.decompress((staging/packed).read_bytes())
                if len(blob)!=record['original_bytes'] or hashlib.sha256(blob).hexdigest()!=record['original_sha256']:
                    raise ValueError('Original gzip SHA mismatch: '+original)
                dst=staging/original;dst.parent.mkdir(parents=True,exist_ok=True);dst.write_bytes(blob)
                if original in verified and verified[original]!=record['original_sha256']:raise ValueError('Conflicting original')
                verified[original]=record['original_sha256']
            receipts.append({'archive':archive.name,'outer_sha256':digest(archive),'manifest':manifest})
    for name,h in verified.items():
        dst=PROJECT_ROOT/name
        if dst.is_symlink() or not dst.resolve().is_relative_to(PROJECT_ROOT.resolve()):raise ValueError('Destination escape')
        if dst.exists() and (not dst.is_file() or digest(dst)!=h):raise FileExistsError('Refusing different-file overwrite: '+str(dst))
    for name in verified:
        dst=PROJECT_ROOT/name
        if not dst.exists():dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(staging/name,dst)
receipt_dir=PROJECT_ROOT/'notebook_import_receipts';receipt_dir.mkdir(exist_ok=True)
(receipt_dir/'IMPORTED_PACKAGES.json').write_text(json.dumps(receipts,indent=2)+'\n')
print({'imported_packages':len(receipts),'verified_payload_files':len(verified),'destination':str(PROJECT_ROOT)})


## Pinned CPU dependencies and durable command logs

No PyTorch or GPU is required. Use the delivered minimal requirement file; changed numerical versions may invalidate cache fingerprints. The runner refuses a mismatched frozen data/code/config manifest. Logs, nonzero exits and partial status remain visible. A partial invocation must not be labelled COMPLETED.


In [ ]:
requirements=PROJECT_ROOT/'requirements-stage3g.txt'
if not requirements.is_file():raise FileNotFoundError('Missing required Stage3G pinned dependencies: '+str(requirements))
ENV_ROOT=Path('/content/wessex_stage3g_cpu_env')
if not (ENV_ROOT/'bin/python').is_file():subprocess.run([sys.executable,'-m','venv',str(ENV_ROOT)],check=True)
PYTHON=str(ENV_ROOT/'bin/python')
subprocess.run([PYTHON,'-m','pip','install','-r',str(requirements)],check=True)
CPU_ENV=dict(os.environ,PYTHONPATH=str(PROJECT_ROOT/'src'),OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1')
LOG_DIR=PROJECT_ROOT/'notebook_logs';LOG_DIR.mkdir(exist_ok=True)
CONFIG=f'configs/{VERSION}/full.json'

def run_logged(arguments,label,allowed_partial=False):
    path=LOG_DIR/(label+'.log')
    with path.open('a',buffering=1) as log:
        log.write('\nSTART '+datetime.datetime.now(datetime.timezone.utc).isoformat()+' '+repr(arguments)+'\n')
        log.flush();os.fsync(log.fileno())
        process=subprocess.Popen([PYTHON]+list(arguments),cwd=PROJECT_ROOT,env=CPU_ENV,stdout=log,stderr=subprocess.STDOUT)
        last=0
        while process.poll() is None:
            time.sleep(2)
            current=path.read_text(errors='replace')
            if len(current)>last:print(current[last:],end='');last=len(current)
        current=path.read_text(errors='replace')
        if len(current)>last:print(current[last:],end='')
        log.write('EXIT '+str(process.returncode)+'\n');log.flush();os.fsync(log.fileno())
    if process.returncode and not(allowed_partial and process.returncode==3):
        raise RuntimeError(f'Command failed ({process.returncode}); preserved log {path}')
    return process.returncode

run_logged(['-c',"from wastewater.stage3g_experiment import load_config; import json; c,s=load_config('"+CONFIG+"'); print(json.dumps({'frozen_manifest_verified':True,'fingerprint':s,'physical_mapping_verified':c['physical_mapping_verified'],'verified_real_time':c['verified_real_time']},indent=2))"],'01_frozen_input_validation')
run_logged(['-m','unittest','discover','-s','tests','-p','test_stage3g*.py','-v'],'02_bounded_software_checks')


## Source-only selection, independent reliability cross-fitting, and smoke

Source fitting is restricted to 2016–2021. Source parameter and honest risk-validation pipelines use training through 2020 and the two fixed 2021 pseudo-deployments. In each outer fold, base fitting, risk training, threshold calibration and held-out evaluation groups are distinct. Final risk operating thresholds are taken from cross-fitted source scores, never target deployment labels.

The smoke has SiteIDs 13013/13015, the first existing 2022 deployment, all budgets and horizons, every fixed source composition, and the prescribed B30 shuffle ablation. Resume reuses only validated completed outputs. This operation may use archived source caches; the runner reports verified reuse separately from calculation.


In [ ]:
run_logged(['-m','wastewater.stage3g_experiment','--config',CONFIG,'--mode','prepare-source'],'03_source_only_selection')
run_logged(['-m','wastewater.stage3g_experiment','--config',CONFIG,'--mode','smoke'],'04_two_site_full_flow_smoke')
smoke_status=PROJECT_ROOT/f'runs/{VERSION}/smoke/RUN_STATUS.json'
if not smoke_status.is_file():raise FileNotFoundError('Smoke completion receipt is missing.')
print(json.dumps(json.loads(smoke_status.read_text()),indent=2))


## Exact full continuation and evaluation

Set `RUN_FULL_FROZEN=True` above only to complete/resume this same approved development matrix. It does not authorize a new experiment, protected facilities or final-year evaluation. A disk-verified complete full run is required before metric recalculation. Confidence intervals describe this reused SiteID development design, not independent physical-facility generalization.


In [ ]:
if RUN_FULL_FROZEN:
    run_logged(['-m','wastewater.stage3g_experiment','--config',CONFIG,'--mode','resume'],'05_frozen_full_resume',allowed_partial=True)
full_status=PROJECT_ROOT/f'runs/{VERSION}/full/RUN_STATUS.json'
FULL_COMPLETED=False
if full_status.is_file():
    status=json.loads(full_status.read_text());print(json.dumps(status,indent=2))
    FULL_COMPLETED=status.get('status')=='COMPLETED' and status.get('disk_verified_completed')==status.get('expected')
if FULL_COMPLETED:
    run_logged(['scripts/summarize_stage3g.py','--config',CONFIG,'--run',f'runs/{VERSION}/full','--output',f'reports/{VERSION}/results'],'06_recalculate_saved_predictions')
    run_logged(['scripts/plot_stage3g.py','--results',f'reports/{VERSION}/results','--output',f'figures/{VERSION}'],'07_plot_real_saved_results')
else:
    print('Full disk-verified completion is absent: metric/figure stages were NOT run. Preserve logs and resume the same config.')


## Save/export this session, with truthful status

Export logs and completion receipts in independently openable ZIPs below 24 MiB. Original scientific prediction files stay in the persistent project; the delivery packaging command creates the separately manifested full evidence packages. Session exports below are **not** substitutes for those complete scientific packages. Drive is reported only as used in the current notebook session, and cannot be retrospectively certified for the original run.


In [ ]:
session={'version':VERSION,'session_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'run_full_frozen_requested':RUN_FULL_FROZEN,'full_disk_verified_complete':FULL_COMPLETED,
    'drive_used_in_this_session':USE_GOOGLE_DRIVE,'original_live_drive_validation_claimed':False,
    'protected_and_2024_2025_performance_enabled':False}
(LOG_DIR/'SESSION_STATUS.json').write_text(json.dumps(session,indent=2)+'\n')
EXPORT_DIR=TASK_STORAGE/'session_exports';EXPORT_DIR.mkdir(exist_ok=True)
files=sorted(p for p in LOG_DIR.rglob('*') if p.is_file())
for path in [smoke_status,full_status]:
    if path.is_file():files.append(path)
limit=24*1024**2;parts=[];current=[];size=0
for path in files:
    if path.stat().st_size>=limit:raise ValueError('An individual log is too large; retain it on persistent storage and split explicitly.')
    if size+path.stat().st_size>20*1024**2 and current:parts.append(current);current=[];size=0
    current.append(path);size+=path.stat().st_size
if current:parts.append(current)
for index,part in enumerate(parts,1):
    name=f'Wessex_Stage3G_NotebookSession_{datetime.datetime.now(datetime.timezone.utc):%Y%m%dT%H%M%SZ}_Part{index:02d}.zip'
    archive=EXPORT_DIR/name
    with zipfile.ZipFile(archive,'x',zipfile.ZIP_DEFLATED,compresslevel=9) as z:
        hashes={}
        for path in part:
            member=path.relative_to(PROJECT_ROOT).as_posix();z.write(path,member);hashes[member]=digest(path)
        z.writestr('SESSION_MANIFEST.json',json.dumps({'session':session,'members_sha256':hashes},indent=2)+'\n')
    if archive.stat().st_size>=limit:raise ValueError('Export exceeds24 MiB')
    with zipfile.ZipFile(archive) as z:
        if z.testzip() is not None:raise ValueError('Session ZIP CRC failure')
    print(name,archive.stat().st_size,digest(archive))
